# 1030 41일차

## 1. WSL 배포판

윈도우 안에서 리눅스를 실행하는 WSL에 배포판을 설치하고 골라 쓰는 방법

배포판마다 홈 디렉터리와 설치한 패키지가 따로라, 용도가 다르면 이름을 다르게 해서 따로 만듦

1. `wsl -l -v` : 설치된 배포판 목록. `-l`은 목록, `-v`는 상태와 WSL 버전까지 같이 봄
2. `wsl --install --name furiosa01 -d Ubuntu-22.04` : 배포판 설치
3. `wsl -d furiosa01` : 이름을 지정해 접속

- `--name furiosa01`의 `furiosa01`은 만들 인스턴스에 붙일 이름이라 내가 정함. 생략하면 배포판 이름을 그대로 씀
- `-d Ubuntu-22.04`의 `Ubuntu-22.04`는 설치할 배포판 종류. 쓸 수 있는 목록은 `wsl -l -o`로 봄
- 글자가 여러 개인 옵션은 앞에 `-`를 두 개 붙임. `-install`, `-name`처럼 하나만 쓰면 인수 오류
- `wsl -l -v`에서 이름 앞의 `*`가 기본 배포판 표시. 이름 없이 `wsl`만 치면 기본 배포판으로 들어감
- 설치 후 처음 들어가면 유닉스 사용자 계정과 비밀번호를 만듦. 비밀번호는 입력해도 화면에 표시되지 않음
- 배포판에서 나올 때는 `exit`

## 2. 윈도우 파일 접근

WSL 안에서 윈도우 쪽 파일을 읽고 쓰는 방법

윈도우에서 받은 파일을 리눅스로 옮기거나, 리눅스에서 만든 파일을 윈도우에서 열어 보려고 씀

1. `/mnt/c/Users/Admin` : 윈도우 C 드라이브가 마운트된 경로. `/mnt/c`가 `C:\`에 해당
2. `explorer.exe .` : 지금 디렉터리를 윈도우 탐색기로 엶. 끝의 `.`이 지금 디렉터리

- 윈도우 프로그램은 `.exe`까지 붙여야 실행됨. `explorer`만 쓰면 명령을 찾지 못함
- 탐색기가 열리면 윈도우에서 그 폴더로 파일을 끌어다 놓을 수 있음

#### cf) Zone.Identifier

- 인터넷에서 받은 파일을 윈도우에서 복사하면 `id_rsa_15:Zone.Identifier`처럼 `:Zone.Identifier`가 붙은 파일이 같이 따라옴
- 파일을 외부에서 받았다고 윈도우가 표시해 두는 메타데이터라 리눅스에서는 쓰지 않음
- 지워도 원래 파일에는 영향이 없음

## 3. SSH 키 접속

비밀번호 대신 키 파일로 원격 서버에 접속하는 방법

비밀번호를 입력하지 않고 접속할 수 있고, 긴 무작위 값이라 비밀번호보다 추측하기 어려움

1. `chmod 600 id_rsa_15` : 키 파일을 소유자만 읽고 쓸 수 있게 바꿈
2. `ssh -i .ssh/id_rsa_15 -p <포트> root@<서버 주소>` : 키 파일을 지정해 접속

- `chmod 600`의 `6`은 소유자에게 읽기·쓰기, 뒤의 `0 0`은 그룹과 나머지 사용자에게 권한 없음
- `-i .ssh/id_rsa_15`의 `id_rsa_15`는 개인 키 파일. 짝이 되는 공개 키는 서버에 등록돼 있어야 함
- `-p <포트>`는 접속할 포트. ssh의 기본 포트는 22라서 다를 때만 지정
- `root@<서버 주소>`는 `접속할 계정@서버 주소`. 포트와 주소는 공개 저장소라 가림
- 키 파일 권한이 느슨하면 ssh가 그 키를 쓰지 않고 접속을 거부함
- 처음 접속하는 서버는 호스트 지문을 보여주고 `yes`를 받아야 연결됨. 한 번 수락하면 `~/.ssh/known_hosts`에 저장돼 다음부터 묻지 않음
- 접속을 끊을 때는 `exit`

#### cf) 포트 매핑

- 접속할 때 치는 포트 번호가 서버 안에서 열려 있는 포트와 다를 수 있음
- 바깥에서 받은 포트를 서버 안의 ssh 포트(22)로 연결해 두기 때문
- 서비스를 띄울 포트를 따로 열어 두면 그 포트도 같은 식으로 바깥 포트를 통해 접근함

## 4. furiosa-smi

NPU의 상태를 확인하는 명령

학습이나 추론을 돌리기 전에 장치가 살아 있는지와 남은 메모리를 확인하려고 씀. GPU의 `nvidia-smi`와 같은 자리

1. `furiosa-smi status` : 장치 상태
2. `furiosa-smi info` : 장치 상세 정보
3. `furiosa-smi ps` : NPU를 쓰고 있는 프로세스 목록
4. `furiosa-smi topo` : 장치 사이의 연결 구조
5. `furiosa-smi version` : 드라이버와 도구 버전

- `status`는 Arch(아키텍처), Device(장치 이름), Liveness(살아 있는지), Memory(쓴 양/전체), Core Utilization(코어별 사용률)을 표로 보여줌
- `info`는 Firmware(펌웨어 버전), Temp.(온도), Power(전력), PCI-BDF(PCI 주소)를 보여줌
- 코어가 여러 개라 사용률이 코어마다 따로 나옴
- 쓰고 있는 프로그램이 없으면 메모리와 코어 사용률이 비어 있음
- 한 번 찍고 끝나므로 계속 보려면 `watch`와 같이 씀

## 5. watch

명령을 일정 간격으로 다시 실행해 화면을 갱신하는 명령

장치 사용률처럼 계속 바뀌는 값을 한 번씩 다시 치지 않고 보려고 씀

1. `watch -n 1 furiosa-smi status` : 1초마다 `furiosa-smi status`를 다시 실행

- `-n 1`의 `1`이 간격(초). `-n`을 빼면 기본 2초
- 화면 맨 위에 간격과 실행한 명령, 호스트 이름, 시각이 나옴
- 매번 화면을 새로 그려서 직전 결과는 남지 않음
- 끝낼 때는 `Ctrl + C`

## 6. 서버 사양 확인

리눅스에서 CPU와 메모리 사양을 보는 명령

작업을 돌리기 전에 쓸 수 있는 코어 수와 남은 메모리를 확인하려고 씀

1. `lscpu` : CPU 구조와 캐시, NUMA 구성
2. `free -h` : 메모리 사용량. `-h`는 바이트 대신 Gi·Ti처럼 읽기 좋은 단위로 바꿈

- `CPU(s)`는 논리 코어 수이고 `Socket(s)` × `Core(s) per socket` × `Thread(s) per core`
- `Thread(s) per core`가 2면 코어 하나가 스레드 두 개를 돌림
- 캐시는 `L1d`(데이터)·`L1i`(명령어)·`L2`·`L3` 순으로 느려지고 커짐
- NUMA는 CPU와 메모리를 묶은 단위. `NUMA node0 CPU(s)`가 그 노드에 속한 코어 번호이고, 다른 노드의 메모리에 접근하면 더 느림

#### cf) free의 available

- `free`는 아무도 안 쓰는 메모리, `buff/cache`는 커널이 캐시로 잡아 둔 메모리
- 캐시는 필요하면 바로 내주기 때문에 `free`가 작아도 메모리가 부족한 것은 아님
- 실제로 더 쓸 수 있는 양은 `available`이고 보통 `free`보다 큼

## 7. Miniconda 설치

리눅스 서버에 conda를 설치해 파이썬 환경을 따로 관리하는 방법

서버에 깔려 있는 파이썬을 그대로 쓰면 작업마다 필요한 패키지 버전이 서로 얽히기 때문에, 환경을 나눠서 씀

1. `curl -O https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh` : 설치 스크립트를 받음
2. `bash Miniconda3-latest-Linux-x86_64.sh` : 설치 실행
3. `source ~/.bashrc` : 바뀐 설정을 지금 셸에 바로 적용
4. `conda --version` : 설치됐는지 확인

- `curl -O`의 `-O`는 주소 끝에 적힌 파일 이름 그대로 저장. 소문자 `-o`는 저장할 이름을 직접 정하는 옵션이라 다름
- 파일 이름의 `latest`는 최신 버전, `Linux-x86_64`는 리눅스 64비트용이라는 뜻
- 설치 중에 라이선스 동의, 설치 위치, `conda init` 실행 여부를 차례로 물어봄
- 설치 위치는 `[/root/miniconda3] >>>`처럼 기본 경로를 보여주고, `Enter`만 누르면 그 경로로 들어감
- `conda init`에 yes를 하면 `~/.bashrc`에 conda 블록이 추가돼 다음 로그인부터 자동으로 잡힘. `conda init --reverse bash`로 되돌림
- 적용되면 프롬프트 앞에 `(base)`가 붙음
- `which conda`로 실제 설치 경로를 확인함

#### cf) 미니콘다와 아나콘다

- 기능 차이는 크지 않고, 아나콘다는 데이터 분석용 패키지를 미리 깔아 둔 상태로 설치됨
- 미니콘다는 conda와 파이썬만 들어 있어서 필요한 패키지만 받아 씀
- 아나콘다는 규모가 큰 조직에서 쓸 때 유료 라이선스가 필요해 회사·기관 서버에서는 미니콘다를 씀

## 8. conda 환경

작업별로 파이썬과 패키지를 따로 담아 두는 단위

하나의 파이썬에 전부 설치하면 작업마다 필요한 버전이 서로 충돌하기 때문에, 작업별로 나눠서 씀

1. `conda create -n furiosa python=3.11` : `furiosa`라는 환경을 파이썬 3.11로 만듦
2. `conda env list` : 환경 목록과 각 환경의 경로
3. `conda activate furiosa` : 환경 켜기
4. `conda deactivate` : 환경 끄기

- `-n furiosa`의 `furiosa`는 환경 이름이라 내가 정함
- 환경은 conda 설치 경로 아래에 만들어짐. 이름이 `furiosa`면 `/root/miniconda3/envs/furiosa`
- 환경마다 파이썬과 패키지가 따로라 한쪽에서 버전을 바꿔도 다른 환경에 영향이 없음
- 환경을 켜면 프롬프트 앞 표시가 `(base)`에서 그 환경 이름으로 바뀜
- 환경을 지울 때는 `conda env remove -n furiosa`

## 9. 윈도우에서 바로 접속

WSL을 거치지 않고 윈도우 PowerShell에서 원격 서버에 접속하는 방법

키를 WSL 안에만 두면 VS Code로 원격 작업을 할 때 WSL을 한 번 더 거쳐야 해서 단계가 늘어남. 윈도우 쪽에 키와 설정을 두면 한 줄로 접속하고 VS Code도 같은 설정을 읽음

1. `C:/Users/Admin/.ssh/config` : 접속 정보를 미리 적어 두는 파일. 확장자 없음
2. `icacls $key /inheritance:r` : 상위 폴더에서 물려받은 권한을 끊음
3. `icacls $key /grant:r "$($env:USERNAME):(R)"` : 내 계정에만 읽기 권한을 줌
4. `ssh furiosa-npu` : config에 적은 `Host` 이름만으로 접속

config에 적는 내용

```
Host furiosa-npu
    HostName <서버 주소>
    User root
    Port <포트>
    IdentityFile C:/Users/Admin/.ssh/id_rsa_15
    IdentitiesOnly yes
```

- `Host furiosa-npu`의 `furiosa-npu`는 내가 부를 이름이라 내가 정함. `ssh` 뒤에 이 이름을 씀
- `IdentitiesOnly yes` : 여기 적은 키만 쓰고 다른 키는 시도하지 않음. 키가 여러 개일 때 엉뚱한 것을 먼저 내밀어 거부당하는 것을 막음
- 윈도우 ssh는 개인 키 권한이 넓으면 접속을 거부하므로 `icacls`로 좁힘. `/inheritance:r`의 `r`은 상속 제거, `/grant:r`의 `r`은 기존 권한을 지우고 새로 줌, `(R)`은 읽기 권한
- 키는 WSL에서 `cp ~/.ssh/id_rsa_15 /mnt/c/Users/Admin/.ssh/`로 옮김
- 윈도우에서 처음 접속하면 호스트 지문을 다시 물어봄. `known_hosts`가 WSL과 윈도우에 따로 있어서 한쪽에서 수락한 것이 다른 쪽에 적용되지 않음

#### cf) VS Code로 원격 폴더 열기

1. `Remote - SSH` 확장 설치
2. `Ctrl + Shift + P` → `Connect to Host` → 목록에서 호스트 선택
3. `Open Folder` → `/root/` : 서버 쪽 작업 폴더를 엶

- VS Code도 같은 `config`를 읽어서 호스트 목록에 `Host` 이름이 그대로 뜸. 주소나 포트를 따로 입력하지 않음
- `Ctrl + Shift + P`는 VS Code의 명령 팔레트. 메뉴를 찾지 않고 명령 이름으로 실행함
- `Open Folder`에 적는 경로는 내 컴퓨터가 아니라 서버 쪽 경로. `/root`는 root 계정의 홈 디렉터리이고, 일반 사용자의 홈은 `/home/<계정 이름>`이라 자리가 다름
- 폴더를 열면 탐색기와 터미널이 모두 서버 쪽을 가리켜서, 파일을 옮기지 않고 서버에서 바로 편집함